In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler

datasets = {
    "full": "../../data/processed/ProcessedData.csv",
    "pca": "../../data/reduced/dataPCA.csv",
    "fs": "../../data/reduced/dataFS.csv"
}

outDir = "../../results/dbscan"

In [7]:
eps = 0.8          
minSamples = 10

def runDbscan(datasetName, df):
    
    dbscan = DBSCAN(
        eps=eps,
        min_samples=minSamples,
        n_jobs=-1
    )
    
    labels = dbscan.fit_predict(df)

    uniqueLabels = set(labels)
    numClusters = len(uniqueLabels - {-1})
    noise = np.sum(labels == -1) / len(labels)

    sil = silhouette_score(df[labels != -1], labels[labels != -1])
    db = davies_bouldin_score(df[labels != -1], labels[labels != -1])

    resultData = df.copy()
    resultData["cluster"] = labels
    resultData.to_csv(
        f"{outDir}/dbscan_{datasetName}_clusters.csv",
        index=False
    )

    plotDbscan(df, labels, datasetName)


    return {
        "Dataset": datasetName,
        "Algorithm": "dbscan",
        "eps": eps,
        "min_samples": minSamples,
        "clusters": numClusters,
        "noise_ratio": noise,
        "Silhouette": sil,
        "Davies_Bouldin": db
    }

In [8]:
from sklearn.decomposition import PCA

def plotDbscan(df, labels, datasetName):
    pca2d = PCA(n_components=2, random_state=42)
    X2d = pca2d.fit_transform(df)

    plt.figure(figsize=(6, 5))

    noiseMask = labels == -1
    plt.scatter(
        X2d[noiseMask, 0],
        X2d[noiseMask, 1],
        c="lightgray",
        s=5,
        label="Noise"
    )

    clusterMask = labels != -1
    plt.scatter(
        X2d[clusterMask, 0],
        X2d[clusterMask, 1],
        c=labels[clusterMask],
        cmap="tab10",
        s=5,
        label="Clusters"
    )

    plt.title(f"DBSCAN ({datasetName})")
    plt.legend(markerscale=2)
    plt.tight_layout()
    plt.savefig(f"{outDir}/dbscan_{datasetName}.png", dpi=200)
    plt.close()

In [9]:
results = []

for name, path in datasets.items():
    df = pd.read_csv(path)
    if "Class" in df.columns:
        y = df["Class"].values
        X = df.drop(columns=["Class"])
    else: 
        X = df
    summary = runDbscan(name, X)
    results.append(summary)

In [10]:
summaryData = pd.DataFrame(results)
summaryData.to_csv(f"{outDir}/dbscan_summary.csv", index=False)
print(summaryData)

  Dataset Algorithm  eps  min_samples  clusters  noise_ratio  Silhouette  \
0    full    dbscan  0.8           10        49     0.093027   -0.170570   
1     pca    dbscan  0.8           10        45     0.072394   -0.152683   
2      fs    dbscan  0.8           10        19     0.018332    0.057957   

   Davies_Bouldin  
0        0.895264  
1        0.707757  
2        0.702384  
